In [ ]:
import os

import numpy as np
from aperocore.core import drs_log
from aperocore.constants import load_functions
import sossisse

from pogos.constants.constants import CDict
from pogos.core import pbase
from pogos.core import general
from pogos import tsovore
from pogos import sossisse_wrap
from pogos import sorbet
from sossisse.core import sbase
from sossisse.core import io
from sossisse.core import misc
from sossisse.general import plots
from sossisse.instruments import Instrument


In [ ]:
# TODO: This should be hidden - with only a call to the yaml file

# Define variables
__NAME__ = 'pogos.recipes.pogos_run'
__PACKAGE__ = pbase.__PACKAGE__
__version__ = pbase.__version__
__authors__ = pbase.__authors__
__date__ = pbase.__date__
__release__ = pbase.__release__
# set description
__description__ = 'Run the POGOS recipe'
__inputs__ = ['GLOBAL.YAML_FILE']
    
    
# get parameters
params = load_functions.get_all_params(name=__NAME__,
                                       description=__description__,
                                       inputargs=__inputs__,
                                       param_file_path='GLOBAL.YAML_FILE',
                                       config_list=[CDict],
                                       external_const=pbase.EXTERNAL_CONST)

# get the SOSSISSE sub-dictionary
sossisse_params = params.get('SOSSISSE')
    # convert to independent parameter dictionary
sparams = sossisse_params.as_param_dict()
# make sure yaml file is set
sparams['INPUTS.PARAM_FILE'] = str(params['GLOBAL.YAML_FILE'])
# get instrument class for SOSSISSE
inst = sossisse.get_parameters(__NAME__=__NAME__, __YAML_DICT__=sparams)


# set the function name
func_name = f'{__NAME__}.linear_recon'
# print the splash
misc.sossart()
# get parameters from instrumental parameters
objname = inst.params['INPUTS.OBJECTNAME']
# print the white light curve splash
print(misc.art('Linear Recon ' + objname, 'blue', 'CYAN'))
# -------------------------------------------------------------------------
# load temporary filenames (should be run before science starts)
inst.define_filenames()
# -------------------------------------------------------------------------
# get the stabiblity table file name
wlc_ltbl_file = inst.get_variable('WLC_LTBL_FILE', func_name)
# return if we have the soss_stablity file
if os.path.exists(wlc_ltbl_file) and inst.params['GENERAL.USE_TEMPORARY']:
    msg = 'File {0} exists we skip white light curve step'
    misc.printc(msg.format(wlc_ltbl_file), 'info')

# Update constants

In [ ]:
# Here you can update constants
# Do this as follows:
#
# inst.params[PARAM_NAME] = VALUE
#

# Fancy Centering 

This does the fancy centering
- Maybe remove this?
- Or at least move it later (after background correction etc)

In [ ]:
# fancy centering - recalculate trace file
inst.fancy_centering()

# Load the data and apply DQ


In [ ]:
# -------------------------------------------------------------------------
# load the image, error and data quality
cube, err, dq = inst.load_data_with_dq()
# -------------------------------------------------------------------------
# apply the dq to the cube
cube, err = inst.apply_dq(cube, err, dq)

# Remove background

In [ ]:
# remove the background
cube, err = inst.remove_background(cube, err)

# Patch isolated bad pixels

In [ ]:
# for each slice of the cube, isolated bad pixels are interpolated with the
# value of their 4 neighbours.
cube = inst.patch_isolated_bads(cube)

# Remove cosmic rays